# Búsqueda Voraz Primero el Mejor (Greedy Best-First Search)

## Descripción de la actividad

En esta práctica se estudiará el funcionamiento de **Greedy Best-First Search (GBFS)** a partir de la estructura general de búsqueda desarrollada previamente para algoritmos no informados.

El punto de partida será una implementación de **Depth-First Search (DFS)** basada en búsqueda en árboles. A partir de ella se realizarán modificaciones progresivas para:

1. Convertir el algoritmo de **búsqueda en árboles** en un algoritmo de **búsqueda en grafos**.
2. Generalizar el manejo de la frontera mediante una **función de evaluación** $f(n)$.
3. Implementar **Greedy Best-First Search**, utilizando:

$f(n)=h(n)$,

donde $h(n)$ representa una estimación del costo restante desde el nodo $n$ hasta el objetivo.
4. Utilizar la **distancia Manhattan** como función heurística:

$h(n)=|x_n-x_G|+|y_n-y_G|$.

5. Registrar el tamaño de la frontera durante la ejecución para comparar experimentalmente el comportamiento de **DFS** y **Greedy Best-First Search**.

El problema se desarrollará sobre un entorno bidimensional tipo **GridWorld**. El agente deberá desplazarse desde un estado inicial hasta un estado objetivo mediante movimientos en las cuatro direcciones cardinales:

- izquierda,
- derecha,
- arriba,
- abajo.

Como reto final, se extenderá el problema para incorporar **celdas no transitables u obstáculos**, identificando qué componente de la formulación del problema debe modificarse para representar correctamente este nuevo escenario.

---

**Asignatura:** Inteligencia Artificial (0406)  
**Grupo:** 01  
**Semestre:** 2027-1  
**Fecha de entrega:** 20 de octubre de 2026

## 1. Definición del escenario

Retomaremos el mismo entorno **GridWorld** utilizado en la actividad anterior. El agente se desplaza sobre una cuadrícula de $3\times3$, iniciando en la posición $(1,1)$ y buscando alcanzar el estado objetivo $(3,3)$.

En esta primera etapa **no se modificarán las características del problema**. Esto permitirá comparar distintas estrategias de búsqueda sobre exactamente el mismo espacio de estados.

Ejecute la siguiente celda para definir los parámetros básicos del entorno.

In [2]:
# Parámetros del entorno GridWorld

# Tamaño de la cuadrícula
N = 3

# Acciones disponibles para el agente
# Cada acción corresponde a un desplazamiento (dx, dy)
acciones = {
    "izquierda": (-1, 0),
    "abajo": (0, -1),
    "arriba": (0, 1),
    "derecha": (1, 0)
}

# Estado inicial del agente
estado_inicial = (1, 1)

# Estado objetivo
estado_objetivo = (N, N)

## Definición del problema

El problema de búsqueda se describe mediante:

- una **función sucesora**, que determina las acciones válidas y los estados alcanzables;
- una **prueba de objetivo**, que verifica si se ha alcanzado la meta;
- una **función de costo**, que asigna un costo a cada transición.

En este GridWorld:

$
\text{SUCESOR-FN}(s)=\{(a,s')\},
\qquad
c(s,a,s')=1.
$

Por tanto, el costo acumulado $g(n)$ coincide con el número de movimientos realizados.

In [3]:
# Funciones del problema
def es_valido(x, y):
    return 1 <= x <= N and 1 <= y <= N

def sucesor_fn(estado):
    x, y = estado
    sucesores = []

    for accion, (dx, dy) in acciones.items():
        nuevo_x = x + dx
        nuevo_y = y + dy

        if es_valido(nuevo_x, nuevo_y):
            sucesores.append((accion, (nuevo_x, nuevo_y)))

    return sucesores

def test_objetivo(estado):
    return estado == estado_objetivo

def costo_accion(estado, accion, estado_sucesor):
    return 1

## Problema de búsqueda

Un problema de búsqueda puede representarse mediante cuatro componentes:

1. **Estado inicial** $s_0$.
2. **Función sucesora** $\text{SUCESOR-FN}(s)$.
3. **Prueba de objetivo**.
4. **Función de costo** $c(s,a,s')$.

A partir de estos elementos, el algoritmo explora el espacio de estados para encontrar un camino desde $s_0$ hasta un estado objetivo.

En esta implementación agruparemos estos componentes en una sola estructura.

In [4]:
# Definición del problema
problema = {
    "estado_inicial": estado_inicial,
    "funcion_sucesora": sucesor_fn,
    "test_objetivo": test_objetivo,
    "costo_accion": costo_accion
}

## Representación de nodos

Es importante distinguir entre un **estado** del problema y un **nodo de búsqueda**.

Un estado representa una configuración del entorno, mientras que un nodo almacena además información sobre el camino seguido para alcanzarlo.

Cada nodo contendrá:

- **estado**: estado representado;
- **padre**: nodo desde el cual fue generado;
- **acción**: acción aplicada;
- **costo_camino**: costo acumulado $g(n)$;
- **profundidad**: número de transiciones desde el nodo inicial.

Esta información permitirá reconstruir posteriormente la solución encontrada.

In [5]:
def hacer_nodo(estado, padre, accion, costo_camino, profundidad):
    return {
        "estado": estado,
        "padre": padre,
        "accion": accion,
        "costo_camino": costo_camino,
        "profundidad": profundidad
    }

## Operaciones sobre los nodos

Durante la búsqueda se realizan dos operaciones fundamentales sobre los nodos:

- **Expandir un nodo:** utiliza la función sucesora para generar nuevos nodos a partir de las transiciones válidas.
- **Reconstruir la solución:** sigue las referencias al nodo padre desde el objetivo hasta el nodo inicial.

Al expandir un nodo, el costo acumulado y la profundidad se actualizan como:

$g(n') = g(n) + c(s,a,s')$

$d(n') = d(n) + 1$

La referencia al nodo padre permite reconstruir posteriormente la secuencia de acciones que forma la solución.

In [6]:
def expandir(nodo, problema):
    sucesores = []

    for accion, estado in problema["funcion_sucesora"](nodo["estado"]):

        nuevo_nodo = hacer_nodo(
            estado,
            nodo,
            accion,
            nodo["costo_camino"] + problema["costo_accion"](
                nodo["estado"],
                accion,
                estado
            ),
            nodo["profundidad"] + 1
        )

        sucesores.append(nuevo_nodo)

    return sucesores


def solucion(nodo):
    camino = []

    while nodo["padre"] is not None:
        camino.append(
            (nodo["padre"]["estado"], nodo["accion"])
        )
        nodo = nodo["padre"]

    camino.reverse()
    return camino

## Búsqueda en árboles

El algoritmo mantiene una **frontera** con los nodos generados que aún no han sido expandidos.

En cada iteración:

1. extrae un nodo de la frontera;
2. verifica si corresponde al objetivo;
3. si no es así, lo expande y agrega sus sucesores.

La estrategia de búsqueda depende de cómo se administra la frontera: FIFO para BFS, LIFO para DFS o prioridad por costo para UCS.

En esta versión se utiliza **búsqueda en árboles**, por lo que un mismo estado puede generarse más de una vez. Posteriormente se modificará el algoritmo para realizar **búsqueda en grafos**.

In [7]:
# Algoritmo general de búsqueda en árboles
def busqueda_arbol(problema, frontera):

    frontera.put([
        hacer_nodo(
            problema["estado_inicial"],
            None,
            None,
            0,
            0
        )
    ])

    while frontera.pendientes():

        nodo = frontera.get()

        if problema["test_objetivo"](nodo["estado"]):
            return solucion(nodo)

        frontera.put(expandir(nodo, problema))

    return None

## Actividad 1 — DFS como punto de partida

La estrategia de búsqueda depende de cómo se administra la **frontera**.

En **Depth-First Search (DFS)**, la frontera se comporta como una pila **LIFO** (*Last-In, First-Out*): el último nodo insertado es el primero en extraerse.

### Instrucciones

1. Implemente una clase `FronteraDFS` utilizando una lista de Python.
2. Mantenga la interfaz:
   - `pendientes()`
   - `get()`
   - `put(data)`
3. Ejecute DFS utilizando el algoritmo `busqueda_arbol`.

```python
f = FronteraDFS()
sol = busqueda_arbol(problema, f)
list(sol)
```

Observe el comportamiento de la búsqueda y la solución obtenida. Esta implementación servirá como punto de partida para modificar posteriormente el algoritmo de **búsqueda en árboles** a **búsqueda en grafos**.

In [8]:
class FronteraDFS:

    def __init__(self):
        self.pila = []

    def pendientes(self):
        return len(self.pila)

    def get(self):
        return self.pila.pop()

    def put(self, nodos):
        for nodo in nodos:
            self.pila.append(nodo)

In [9]:
frontera_dfs = FronteraDFS()
solucion_dfs = busqueda_arbol(problema, frontera_dfs)

In [10]:
for item in solucion_dfs:
    print(item)

((1, 1), 'derecha')
((2, 1), 'derecha')
((3, 1), 'arriba')
((3, 2), 'arriba')


## Actividad 2 — Implementar búsqueda en grafos

La implementación actual realiza **búsqueda en árboles**, por lo que un mismo estado puede aparecer y expandirse más de una vez.

Modifique el algoritmo para implementar **búsqueda en grafos**, siguiendo lo más cercanamente posible el siguiente procedimiento:

1. Inicializar:

$cerrado \leftarrow \emptyset$

2. Insertar en la frontera un nodo correspondiente al estado inicial.

3. Mientras la frontera no esté vacía:

   - extraer un nodo de la frontera;
   - aplicar el **test objetivo**;
   - si el estado del nodo no pertenece a `cerrado`:
     - agregar su estado a `cerrado`;
     - expandir el nodo;
     - insertar todos sus sucesores en la frontera.

4. Si la frontera queda vacía, devolver fallo.

### Instrucciones

- Implemente una nueva función llamada `busqueda_grafo`.
- Utilice un `set` de Python para representar `cerrado`.
- Mantenga sin cambios la clase `FronteraDFS`.
- Ejecute DFS utilizando primero `busqueda_arbol` y después `busqueda_grafo`.

### Comparación

Observe ambas ejecuciones y responda:

1. ¿Ambas encuentran una solución?
2. ¿La solución obtenida es la misma?
3. ¿Qué ocurre con los estados que pueden alcanzarse por más de un camino?
4. ¿Qué ventaja introduce el conjunto `cerrado`?

In [17]:
def busqueda_grafo(problema, frontera):
    cerrado = set()
    
    frontera.put([
    hacer_nodo(
        problema["estado_inicial"],
        None,
        None,
        0,
        0
    )
    ])

    while frontera.pendientes():

        nodo = frontera.get()

        if problema["test_objetivo"](nodo["estado"]):
            return solucion(nodo)

        if not nodo["estado"] in cerrado:
            cerrado.add(nodo["estado"])
            frontera.put(expandir(nodo, problema))
       
    return None

In [18]:
## VALIDA TU IMPLEMENTACIÓN

f = FronteraDFS()
sol = busqueda_grafo(problema, f)

resultado_esperado = [
    ((1, 1), "derecha"),
    ((2, 1), "derecha"),
    ((3, 1), "arriba"),
    ((3, 2), "arriba")
]

assert sol == resultado_esperado, "La solución obtenida no coincide con la esperada."

print("La búsqueda en grafos funciona correctamente.")

La búsqueda en grafos funciona correctamente.


## Actividad 3 — Generalizar la frontera mediante $f(n)$

En las estrategias de búsqueda con prioridad, el siguiente nodo que se extrae de la frontera depende de una **función de evaluación** $f(n)$.

Por ejemplo:

$f_{\text{UCS}}(n)=g(n)$

donde $g(n)$ es el costo acumulado del camino hasta el nodo.

### Instrucciones

1. Modifique la implementación de la frontera con prioridad para que reciba una función $f(n)$.
2. Al insertar cada nodo, calcule su prioridad mediante:

$\text{prioridad}=f(n)$

3. La frontera debe extraer siempre el nodo con el menor valor de $f(n)$.
4. Mantenga la misma interfaz utilizada anteriormente:
   - `pendientes()`
   - `get()`
   - `put(nodos)`

5. Verifique inicialmente el funcionamiento definiendo:

$f(n)=g(n)$

de modo que la estrategia obtenida sea equivalente a **Uniform Cost Search (UCS)**.

Esta generalización permitirá reutilizar la misma frontera con diferentes funciones de evaluación en las siguientes actividades.

In [21]:
import heapq

class FronteraPrioridad:
    
    def __init__(self, fn):
         self.heap = []
         self.contador = 0
         self.fn = fn

    def pendientes(self):
        return len(self.heap)

    def get(self):
        nodo = heapq.heappop(self.heap)
        return nodo[2]

    def put(self,data):
        for nodo in data:
            prioridad = self.fn(nodo)
            heapq.heappush(self.heap,(prioridad,self.contador,nodo))
            self.contador += 1

In [22]:
def f_ucs(nodo):
    return nodo["costo_camino"]

f = FronteraPrioridad(f_ucs)
sol = busqueda_grafo(problema, f)

list(sol)

[((1, 1), 'arriba'),
 ((1, 2), 'arriba'),
 ((1, 3), 'derecha'),
 ((2, 3), 'derecha')]

## Actividad 4 — Implementar Greedy Best-First Search

En **Greedy Best-First Search (GBFS)**, la prioridad de cada nodo se determina únicamente mediante una función heurística:

$f(n)=h(n)$

Para este GridWorld utilizaremos como heurística la **distancia Manhattan** entre el estado actual $(x,y)$ y el estado objetivo $(x_G,y_G)$:

$h(n)=|x-x_G|+|y-y_G|$

### Instrucciones

1. Implemente una función `h(nodo)` que calcule la distancia Manhattan desde el estado del nodo hasta el objetivo.
2. Defina la función de evaluación de Greedy como:

$f_{\text{GBFS}}(n)=h(n)$

3. Utilice la clase `FronteraPrioridad` desarrollada en la actividad anterior.
4. Ejecute `busqueda_grafo` utilizando esta nueva función de evaluación.
5. Observe el orden en que GBFS orienta la búsqueda hacia el estado objetivo.

### Reflexión

- ¿Qué información utiliza GBFS para seleccionar el siguiente nodo?
- ¿Considera el costo acumulado $g(n)$?
- ¿Qué diferencia conceptual existe entre $f_{\text{UCS}}(n)=g(n)$ y $f_{\text{GBFS}}(n)=h(n)$?

In [26]:
def h(nodo):
    x = nodo["estado"][0]
    y = nodo["estado"][1]
    xg = estado_objetivo[0]
    yg = estado_objetivo[1]

    return abs(x - xg) + abs(y - yg)
    

In [27]:
def f_gbfs(nodo):
    return h(nodo)

In [28]:
# VALIDA TU IMPLEMENTACIÓN

f = FronteraPrioridad(f_gbfs)
sol = busqueda_grafo(problema, f)

resultado_esperado = [
    ((1, 1), "arriba"),
    ((1, 2), "arriba"),
    ((1, 3), "derecha"),
    ((2, 3), "derecha")
]

assert sol == resultado_esperado, "La solución obtenida no coincide con la esperada."

print("Greedy Best-First Search funciona correctamente.")

Greedy Best-First Search funciona correctamente.


## Actividad 5 — Registrar el tamaño de la frontera

Además de encontrar una solución, podemos instrumentar un algoritmo para observar su comportamiento durante la ejecución.

En esta actividad registraremos el número de nodos almacenados en la **frontera**:

$|\text{frontera}|$

### Instrucciones

1. Modifique `FronteraDFS` y `FronteraPrioridad` para incorporar una lista llamada `tamanos`.
2. Después de cada operación `get()` o `put()`, registre el número de nodos pendientes en la frontera.
3. Ejecute `busqueda_grafo` utilizando:
   - DFS;
   - Greedy Best-First Search.
4. Muestre las secuencias almacenadas en `tamanos`.

### Reflexión

- ¿Qué representa cada valor registrado?
- ¿Por qué el tamaño de la frontera aumenta y disminuye durante la búsqueda?
- ¿El tamaño de la frontera representa por sí solo toda la memoria utilizada por una búsqueda en grafos? ¿Qué otra estructura utiliza el algoritmo?

In [29]:
class FronteraDFS:

    def __init__(self):
        self.pila = []
        self.tamanos = []

    def pendientes(self):
        return len(self.pila)

    def get(self):
        nodo = self.pila.pop()
        self.tamanos.append(self.pendientes())
        return nodo

    def put(self, nodos):
        for nodo in nodos:
            self.pila.append(nodo)
        self.tamanos.append(self.pendientes())

In [30]:
import heapq

class FronteraPrioridad:
    
    def __init__(self, fn):
         self.heap = []
         self.contador = 0
         self.fn = fn
         self.tamanos = []

    def pendientes(self):
        return len(self.heap)

    def get(self):
        nodo = heapq.heappop(self.heap)
        self.tamanos.append(self.pendientes())
        return nodo[2]

    def put(self,data):
        for nodo in data:
            prioridad = self.fn(nodo)
            heapq.heappush(self.heap,(prioridad,self.contador,nodo))
            self.contador += 1
        self.tamanos.append(self.pendientes())

In [31]:
# DFS
frontera_dfs = FronteraDFS()
sol_dfs = busqueda_grafo(problema, frontera_dfs)

# GBFS
frontera_gbfs = FronteraPrioridad(f_gbfs)
sol_gbfs = busqueda_grafo(problema, frontera_gbfs)

In [32]:
print("DFS :", frontera_dfs.tamanos)
print("GBFS:", frontera_gbfs.tamanos)

DFS : [1, 0, 2, 1, 4, 3, 5, 4, 7, 6]
GBFS: [1, 0, 2, 1, 4, 3, 5, 4, 7, 6]


### Observación

En este escenario pequeño, DFS y GBFS pueden registrar una evolución similar del tamaño de la frontera.

Esto no significa que ambas estrategias sean equivalentes. En la siguiente actividad modificaremos el escenario para observar con mayor claridad las diferencias en su comportamiento.

## Actividad 6 — Comparación experimental de DFS y GBFS

En el GridWorld anterior, DFS y GBFS pueden presentar un comportamiento similar debido al tamaño reducido del problema y al orden en que se generan los sucesores.

Para observar con mayor claridad las diferencias entre ambas estrategias, utilizaremos ahora:

$N=8$

y modificaremos el orden de las acciones. El mismo escenario y el mismo orden de sucesores deberán utilizarse para ambos algoritmos.

### Instrumentación de la búsqueda

Modifique la búsqueda en grafos para registrar, después de cada expansión:

- número de estados expandidos;
- tamaño de la frontera;
- tamaño del conjunto `cerrado`.

Defina además:

$M(t)=|\text{frontera}_t|+|\text{cerrado}_t|$

como una medida experimental del número de elementos almacenados durante la búsqueda.

Ejecute el experimento utilizando:

- DFS;
- Greedy Best-First Search.

Compare para ambas estrategias:

1. número de estados expandidos;
2. longitud de la solución encontrada;
3. tamaño máximo de la frontera;
4. tamaño máximo de `cerrado`;
5. máximo valor observado de $M(t)$.

### Relación con la complejidad espacial

Las cotas estudiadas inicialmente para **búsqueda en árboles** dependen de la estructura del árbol generado. Por ejemplo:

$\text{DFS}=O(bm)$

mientras que GBFS puede requerir, en el peor caso:

$O(b^m)$,

donde $b$ es el factor de ramificación y $m$ la profundidad máxima.

En esta práctica utilizamos **búsqueda en grafos**, por lo que el algoritmo mantiene además un conjunto `cerrado` con los estados ya explorados.

En este caso, resulta más natural expresar el uso de memoria en función del número de estados alcanzables:

$O(|V|)$,

donde $V$ representa el conjunto de estados del espacio de búsqueda.

Para un GridWorld de tamaño $N\times N$, sin obstáculos:

$|V|=N^2$.

Por tanto, para:

$N=8$,

el espacio contiene como máximo:

$|V|=64$

estados diferentes.

Aunque DFS y GBFS tienen la misma cota espacial en términos de $|V|$ para este escenario, esto no significa que almacenen o exploren la misma cantidad de información en una ejecución concreta.

### Reflexión

1. ¿Cuántos de los 64 estados posibles fueron expandidos por DFS y por GBFS?
2. ¿Cuál estrategia alcanzó un mayor tamaño de frontera?
3. ¿Cuál presentó un mayor valor de $M(t)$?
4. Si ambas estrategias tienen una cota espacial $O(|V|)$, ¿por qué los valores observados son diferentes?
5. ¿Qué efecto tuvo la heurística sobre el número de estados expandidos por GBFS?
6. ¿Podemos concluir, a partir de una sola ejecución, que GBFS siempre utilizará menos memoria que DFS? Justifique.

In [41]:
def busqueda_grafo_medida(problema, frontera):

    resultado = {
        "tamanos_frontera": [],
        "tamanos_cerrado": [],
        "memoria_total": [],
        "expandidos": 0,
        "solucion": None
    }
    
    cerrado = set()
    
    frontera.put([
    hacer_nodo(
        problema["estado_inicial"],
        None,
        None,
        0,
        0
    )
    ])

    while frontera.pendientes():

        nodo = frontera.get()

        if problema["test_objetivo"](nodo["estado"]):
            resultado["solucion"] = solucion(nodo)
            return resultado

        if not nodo["estado"] in cerrado:
            cerrado.add(nodo["estado"])
            frontera.put(expandir(nodo, problema))
            resultado["tamanos_frontera"].append(frontera.pendientes())
            resultado["tamanos_cerrado"].append(len(cerrado))
            resultado["memoria_total"].append(frontera.pendientes()+len(cerrado))
            resultado["expandidos"] = resultado["expandidos"] + 1

    resultado["solucion"] = "fallo"
    return resultado

In [42]:
N = 8

# El orden de las acciones se modifica para observar
# diferencias más claras entre DFS y GBFS
acciones = {
    "derecha": (1, 0),
    "arriba": (0, 1),
    "izquierda": (-1, 0),
    "abajo": (0, -1)
}

estado_inicial = (1, 1)
estado_objetivo = (N, N)

problema = {
    "estado_inicial": estado_inicial,
    "funcion_sucesora": sucesor_fn,
    "test_objetivo": test_objetivo,
    "costo_accion": costo_accion
}

In [43]:
# DFS
frontera_dfs = FronteraDFS()

resultado_dfs = busqueda_grafo_medida(
    problema,
    frontera_dfs
)


# GBFS
frontera_gbfs = FronteraPrioridad(f_gbfs)

resultado_gbfs = busqueda_grafo_medida(
    problema,
    frontera_gbfs
)

In [44]:
print("DFS")
print("Nodos expandidos:", resultado_dfs["expandidos"])
print("Longitud solución:", len(resultado_dfs["solucion"]))
print("Máximo frontera:", max(resultado_dfs["tamanos_frontera"]))
print("Máximo cerrado:", max(resultado_dfs["tamanos_cerrado"]))
print("Máximo M(t):", max(resultado_dfs["memoria_total"]))

print()

print("GBFS")
print("Nodos expandidos:", resultado_gbfs["expandidos"])
print("Longitud solución:", len(resultado_gbfs["solucion"]))
print("Máximo frontera:", max(resultado_gbfs["tamanos_frontera"]))
print("Máximo cerrado:", max(resultado_gbfs["tamanos_cerrado"]))
print("Máximo M(t):", max(resultado_gbfs["memoria_total"]))

DFS
Nodos expandidos: 56
Longitud solución: 56
Máximo frontera: 91
Máximo cerrado: 56
Máximo M(t): 147

GBFS
Nodos expandidos: 14
Longitud solución: 14
Máximo frontera: 27
Máximo cerrado: 14
Máximo M(t): 41


## Reto final — GridWorld con obstáculos

Hasta ahora hemos trabajado con una cuadrícula sin restricciones internas. Considere ahora un escenario en el que algunas celdas representan **obstáculos** y no pueden ser atravesadas.

Se proporciona una función que genera automáticamente un GridWorld con:

- dimensiones variables;
- estado inicial;
- estado objetivo;
- conjunto de obstáculos.

### Reto

Adapte la implementación desarrollada durante la práctica utilizando el **menor número posible de cambios**, de forma que DFS y GBFS puedan resolver este nuevo escenario.

Identifique qué componentes deben modificarse y cuáles pueden mantenerse sin cambios.

En particular, determine:

- qué parte de la definición del problema debe considerar los obstáculos;
- si es necesario modificar `busqueda_grafo`;
- si es necesario modificar `FronteraDFS`;
- si es necesario modificar `FronteraPrioridad`;
- si puede mantenerse la heurística Manhattan.

### Reflexión

¿Qué componente de la formulación del problema representa las restricciones impuestas por los obstáculos?

In [ ]:
### CREACIÓN DE ESCENARIOS ALEATORIOS

import random

def crear_escenario(ancho=10, alto=8, prob_muro=0.2, semilla=7, inicio=(1, 1)):
    random.seed(semilla)

    celdas = [
        (x, y)
        for x in range(1, ancho + 1)
        for y in range(1, alto + 1)
        if (x, y) != inicio
    ]

    muros = {
        celda
        for celda in celdas
        if random.random() < prob_muro
    }

    libres = [celda for celda in celdas if celda not in muros]
    meta = random.choice(libres)

    return {
        "ancho": ancho,
        "alto": alto,
        "muros": muros,
        "inicio": inicio,
        "meta": meta
    }

In [ ]:
problema = crear_problema(
    ancho=10,
    alto=8,
    prob_muro=0.2,
    semilla=1984,
    inicio=(1, 1)
)

print("Inicio:", problema["estado_inicial"])
print("Meta:", problema["estado_objetivo"])
print("Obstáculos:", problema["muros"])

In [ ]:
## FUNCIÓN PARA MOSTRAR EL PROBLEMA A RESOLVER

def mostrar(problema, camino=None):
    camino = camino or []

    # La solución tiene pares (estado, acción)
    estados_camino = {estado for estado, accion in camino}

    for y in reversed(range(1, problema["alto"] + 1)):
        fila = ""

        for x in range(1, problema["ancho"] + 1):
            p = (x, y)

            if p == problema["estado_inicial"]:
                fila += "S "
            elif p == problema["estado_objetivo"]:
                fila += "G "
            elif p in problema["muros"]:
                fila += "# "
            elif p in estados_camino:
                fila += "o "
            else:
                fila += ". "

        print(fila)

    print()

In [ ]:
mostrar(problema)

In [ ]:
######## INGRESA TU CÓDIGO AQUÍ


    #############

In [ ]:
######## INGRESA TU CÓDIGO AQUÍ


    #############

In [ ]:
######## INGRESA TU CÓDIGO AQUÍ


    #############